In [ ]:
# === 노트북 공통 preamble (llm_math 패키지 로드) ===
import sys
from pathlib import Path

# llm_math 패키지를 찾을 수 있는 후보 경로들
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 현재 디렉토리의 상위 디렉토리들도 후보에 추가 (notebooks/ 폴더에서 실행 시)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math import 시도
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[주의] llm_math 패키지 로드 실패: {e}")
    print("  GitHub 레포를 클론하고 colab_setup.sh를 실행하세요.")
# === preamble 끝 ===


# Ch 16. Positional Encoding — 순서 정보를 주입하다

> **학습 목표**
> - 트랜스포머가 본래 순서 정보가 없는 이유를 설명한다
> - Sinusoidal, Learned, ALiBi, RoPE의 수학을 이해한다
> - RoPE 회전행렬이 왜 최신 LLM 표준인지 안다

## 16.1 순서 정보 문제

어텐션은 **순열 동등성(permutation equivariant)**: 입력 순서를 바꿔도 출력도 같이 바뀜 → 순서 정보 손실.

"나는 사과를 먹었다" vs "사과를 나는 먹었다" — 어텐션만으로는 구분 불가.

해결: 위치 정보를 임베딩에 더하기. 여러 방법이 제안됨.

## 16.2 Sinusoidal Positional Encoding (원래 트랜스포머)

$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d}}\right), \quad PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d}}\right)$$

- $pos$: 토큰 위치
- $i$: 차원 인덱스
- 주파수가 $10000^{-2i/d}$로 기하급수적으로 변화 → 다양한 스케일에서 위치 인코딩

임베딩에 단순 더함: $\mathbf{x} = \mathbf{e} + PE_{pos}$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def sinusoidal_pe(seq_len, d_model):
    """Sinusoidal Positional Encoding."""
    pe = np.zeros((seq_len, d_model))
    pos = np.arange(seq_len)[:, None]
    div = np.exp(np.arange(0, d_model, 2) * (-np.log(10000) / d_model))
    pe[:, 0::2] = np.sin(pos * div)
    pe[:, 1::2] = np.cos(pos * div)
    return pe

pe = sinusoidal_pe(100, 64)
print(f"PE shape: {pe.shape}")
print(f"PE[0, :8]: {pe[0, :8].round(4)}")
print(f"PE[1, :8]: {pe[1, :8].round(4)}")

# 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
im = axes[0].imshow(pe, cmap='RdBu', aspect='auto')
plt.colorbar(im, ax=axes[0])
axes[0].set_xlabel('Embedding 차원')
axes[0].set_ylabel('Position')
axes[0].set_title('Sinusoidal Positional Encoding')

# 특정 차원의 Value
for i in [0, 4, 16, 32]:
    axes[1].plot(pe[:, i], label=f'dim {i}', linewidth=1.5)
axes[1].set_xlabel('Position')
axes[1].set_ylabel('PE Value')
axes[1].set_title('차원별 PE Pattern')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch16_sinusoidal_pe.png', dpi=100, bbox_inches='tight')
plt.show()


## 16.3 Learned Positional Embedding

BERT, GPT-2는 학습 가능한 위치 임베딩 $\mathbf{E}_{pos} \in \mathbb{R}^{T_{\max} \times d}$ 사용.

장점: 데이터에서 최적 위치 표현 학습.
단점: 학습 길이 $T_{\max}$ 넘어가면 사용 불가 (외삽 불가).


In [ ]:
# Learned PE (PyTorch)
import torch
import torch.nn as nn

max_len, d = 512, 64
learned_pe = nn.Embedding(max_len, d)

# 위치 인덱스로 임베딩 조회
positions = torch.arange(10).unsqueeze(0)  # (1, 10)
pe_emb = learned_pe(positions)  # (1, 10, d)
print(f"Learned PE: {pe_emb.shape}")
print(f"초깃값은 랜덤, 학습하면서 최적화됨")


## 16.4 ALiBi — 외삽 가능한 위치 인코딩

**ALiBi** (Press et al., 2022)는 위치 임베딩을 더하지 않고, 어텐션 점수에 편향 추가:

$$\mathrm{softmax}\left(QK^\top + m \cdot (i - j)\right)$$

- $m$: 헤드별 기울기 (기하급수적으로 감소)
- $(i - j)$: 상대적 거리 (음수)
- 먼 토큰일수록 점수 감소

**장점**: 학습 길이보다 긴 시퀀스로 외삽 가능.


In [ ]:
# ALiBi 마스크
def alibi_bias(n_heads, seq_len):
    """ALiBi Bias 행렬: (n_heads, T, T)."""
    # Head별 그래디언트: 2^(-8 * h / n_heads), h=1..n_heads
    slopes = 1.0 / (2 ** (8 * torch.arange(1, n_heads + 1).float() / n_heads))  # (n_heads,)
    # Distance 행렬: (T, T), relative[i, j] = j - i (양수이면 미래, 음수이면 과거)
    positions = torch.arange(seq_len)
    relative = positions[None, :] - positions[:, None]  # (T, T)
    # broadcast: (n_heads, 1, 1) * (1, T, T) -> (n_heads, T, T)
    bias = slopes[:, None, None] * relative[None, :, :]
    return bias

n_heads, seq_len = 8, 16
bias = alibi_bias(n_heads, seq_len)
print(f"ALiBi bias: {bias.shape}")

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for h in range(n_heads):
    ax = axes[h // 4, h % 4]
    im = ax.imshow(bias[h].numpy(), cmap='RdBu_r')
    ax.set_title(f'Head {h} (slope={1/(2**(8/n_heads))**(h+1):.4f})')
    plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.savefig('../figures/ch16_alibi.png', dpi=100, bbox_inches='tight')
plt.show()
print("각 Head가 다른 그래디언트로 Distance에 따른 Decrease Application.")


## 16.5 RoPE (Rotary Position Embedding) — LLaMA 표준

**RoPE** (Su et al., 2021)는 위치 $m$에서 쿼리/키를 회전:

$$\mathrm{RoPE}(\mathbf{x}, m) = R_m \mathbf{x}$$

여기서 $R_m$은 블록 대각 회전 행렬:

$$R_m = \mathrm{diag}\left(R(m\theta_1), R(m\theta_2), \ldots, R(m\theta_{d/2})\right)$$

$$R(\theta) = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}, \quad \theta_i = 10000^{-2i/d}$$

**핵심 성질**:
$$\langle \mathrm{RoPE}(\mathbf{q}, m), \mathrm{RoPE}(\mathbf{k}, n) \rangle = \langle \mathbf{q}, \mathbf{k} \rangle \text{ (회전 후)}$$

→ 내적이 **상대적 거리 $m - n$**에만 의존. 절대 위치가 아닌 상대적 위치 인코딩.


In [ ]:
# RoPE 구현
def rotate_half(x):
    x1, x2 = x[..., :x.shape[-1] // 2], x[..., x.shape[-1] // 2:]
    return torch.cat([-x2, x1], dim=-1)

def apply_rope(x, positions, theta_base=10000.0):
    """x: (B, H, T, d), positions: (T,)."""
    d = x.shape[-1]
    # theta_i = 1 / theta_base^(2i/d)
    freqs = 1.0 / (theta_base ** (torch.arange(0, d, 2).float() / d))  # (d/2,)
    # angles: (T, d/2)
    angles = positions[:, None] * freqs[None, :]
    # cos, sin: (T, d) — 각 차원 Iteration
    cos = torch.cos(angles).repeat_interleave(2, dim=-1)  # (T, d)
    sin = torch.sin(angles).repeat_interleave(2, dim=-1)  # (T, d)
    # (1, 1, T, d)
    cos = cos[None, None, :, :]
    sin = sin[None, None, :, :]
    return x * cos + rotate_half(x) * sin

# Test
d = 8
T = 4
x = torch.randn(1, 1, T, d)
positions = torch.arange(T)
x_rope = apply_rope(x, positions)
print(f"입력: {x.shape}")
print(f"RoPE Application 후: {x_rope.shape}")

# 핵심 성질 검증: 상대적 거리만 의존
# q at pos 0, k at pos 1 vs q at pos 1, k at pos 2 (같은 거리 1)
q1 = torch.randn(1, 1, 1, d)
k1 = torch.randn(1, 1, 1, d)
pos_q1 = torch.tensor([0])
pos_k1 = torch.tensor([1])
pos_q2 = torch.tensor([1])
pos_k2 = torch.tensor([2])

dot1 = (apply_rope(q1, pos_q1) * apply_rope(k1, pos_k1)).sum()
dot2 = (apply_rope(q1, pos_q2) * apply_rope(k1, pos_k2)).sum()
print(f"\nRoPE 상대적 Distance 성질 Validation:")
print(f"  (q@0, k@1) Inner Product = {dot1.item():.6f}")
print(f"  (q@1, k@2) Inner Product = {dot2.item():.6f}")
print(f"  Difference: {abs(dot1 - dot2).item():.2e} (≈0, 같은 상대적 Distance)")


## 16.6 RoPE 시각화

2D에서 RoPE가 어떻게 동작하는지 시각화.


In [ ]:
# 2D RoPE 시각화
fig, ax = plt.subplots(figsize=(8, 8))
d = 2
theta = 1.0  # 단순화

# 원본 벡터
v = np.array([2.0, 1.0])
ax.quiver(0, 0, v[0], v[1], angles='xy', scale_units='xy', scale=1,
          color='blue', linewidth=3, label='원본')

# 위치 1, 2, 3, 4에서 회전
for m in [1, 2, 3, 4]:
    angle = m * theta
    R = np.array([[np.cos(angle), -np.sin(angle)],
                  [np.sin(angle),  np.cos(angle)]])
    v_rot = R @ v
    ax.quiver(0, 0, v_rot[0], v_rot[1], angles='xy', scale_units='xy', scale=1,
              color='red', alpha=0.5 + 0.1 * m, linewidth=2,
              label=f'pos={m} (Rotation {np.degrees(angle):.0f}°)')

ax.set_xlim(-3, 3); ax.set_ylim(-3, 3)
ax.set_aspect('equal')
ax.axhline(0, color='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.5)
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_title('RoPE: Position m에서 Vector를 m·θ 만큼 Rotation')
plt.tight_layout()
plt.savefig('../figures/ch16_rope_rotation.png', dpi=100, bbox_inches='tight')
plt.show()
print("=> Position가 멀어질수록 더 많이 Rotation. 두 Vector의 Inner Product은 Rotation각 Difference(=상대적 Distance)에만 의존.")


## 16.7 긴 문맥 확장

RoPE의 한계: 학습 길이 넘어가면 성능 저하. 해결책:

- **Position Interpolation (PI)**: 위치를 스케일링 ($m \to m \cdot L_{train}/L_{test}$)
- **NTK-aware**: 주파수를 비선형적으로 조정
- **YaRN**: NTK 개선 + 스케일링

LLaMA-3, Qwen 등에서 긴 문맥(128K+) 지원에 사용.


## 16.8 핵심 정리

| 방법 | 외삽 | 사용 모델 |
|---|---|---|
| Sinusoidal | 가능 (이론적) | 원래 Transformer |
| Learned | 불가 | BERT, GPT-2 |
| ALiBi | 가능 | BLOOM |
| RoPE | 부분 (스케일링 필요) | LLaMA, Qwen, Mistral |

## 연습문제

1. Sinusoidal PE의 차원 0과 차원 32의 주파수를 계산하고 비교하라.
2. 학습 가능한 PE로 100 위치를 학습하고, 위치 0~50과 51~99의 차이를 시각화하라.
3. ALiBi의 헤드별 기울기 공식 $m_h = 2^{-8h/H}$를 적용하고, 긴 시퀀스에서의 효과를 시뮬레이션하라.
4. RoPE의 "상대적 거리만 의존" 성질을 다양한 위치 쌍에서 검증하라.
5. Position Interpolation의 수식 $m \to m \cdot L_{train}/L_{test}$를 RoPE에 적용하여 외삽 효과를 설명하라.

> 해설: `solutions/ch16_solutions.ipynb`
